# Construire le jeu d'entraînement du service client

Le service client de Biclou & Cie, le fil rouge de la formation, a exporté de son outil de support deux ans de tickets : `tickets.csv`, environ 2 400 messages de clients avec la réponse de l'agent, tels que l'outil les a gardés. Ce notebook en tire le jeu d'entraînement du modèle, au format qu'attend `trl` : chaque ticket utilisable devient une paire (prompt, réponse attendue en JSON), sans doublon, sans donnée personnelle, et sans rien du jeu de test.

1. Explorer l'export
2. Nettoyer les messages
3. Écarter les tickets inutilisables
4. Dédupliquer
5. Retirer les fuites vers le test
6. Mettre au format de `trl` et découper
7. Enregistrer et publier

Pas besoin de GPU : tout tourne en quelques secondes sur le processeur.

## Données

`tickets.csv`, l'export brut ; `test.jsonl`, les 150 tickets que l'équipe a validés pour mesurer le modèle (le TP précédent) ; `charte.md`, la charte de réponse.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-shop-support/tickets.csv
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-shop-support/test.jsonl
%pip install -q datasets

In [ ]:
import html
import json
import re
import unicodedata

import pandas as pd
from datasets import DatasetDict, load_dataset

CATEGORIES = [
  "information",
  "livraison",
  "reparation",
  "retour",
  "location",
  "facturation",
  "compte",
]
PRIORITES = ["basse", "normale", "haute"]
CONSIGNES = f"""Tu es l'assistant du service client de Biclou & Cie, magasins de vélos.
Pour le message du client, réponds uniquement par un objet JSON avec trois clés :
- categorie : une valeur parmi {", ".join(CATEGORIES)} ;
- priorite : basse, normale ou haute ;
- reponse : la réponse à envoyer au client."""
SIGNATURE = "Bonne journée,\nL'équipe Biclou & Cie"
TUTOIEMENT = re.compile(r"\b(tu|te|toi|ton|ta|tes)\b|\bt['’]", re.IGNORECASE)


def respecte_charte(reponse: str) -> bool:
  """Les règles vérifiables de la charte, comme dans le TP précédent."""
  reponse = reponse.strip()
  return (
    reponse.startswith("Bonjour")
    and reponse.endswith(SIGNATURE)
    and not TUTOIEMENT.search(reponse)
    and 40 <= len(reponse.split()) <= 120
  )


test = load_dataset("json", data_files="test.jsonl", split="train")
print(test)

## 1. Explorer l'export

Chargez `tickets.csv` avec `load_dataset` (format `"csv"`), en gardant les cellules vides comme des chaînes vides (`keep_default_na=False`, transmis à pandas). Affichez le jeu, puis, avec `to_pandas()`, la répartition des valeurs de `statut`, `categorie`, `priorite` et `satisfaction`, et quelques messages du canal `formulaire`. Notez ce qui empêcherait d'entraîner un modèle sur ces lignes telles quelles : il y a au moins six problèmes différents.

**Documentation utile**

- [`load_dataset`](https://huggingface.co/docs/datasets/package_reference/loading_methods#datasets.load_dataset)
- [Charger un CSV](https://huggingface.co/docs/datasets/loading#csv)
- [`Dataset.to_pandas`](https://huggingface.co/docs/datasets/package_reference/main_classes#datasets.Dataset.to_pandas)

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `load_dataset("csv", data_files="tickets.csv", split="train", keep_default_na=False)` : sans `split`, `load_dataset` renvoie un `DatasetDict` avec une seule partie, `train`
- `brut.to_pandas()["statut"].value_counts()`, de même pour les autres colonnes
- `df[df["canal"] == "formulaire"]["message"].head(3).tolist()`

```python
brut = load_dataset("csv", data_files=..., split="train", keep_default_na=False)
print(brut)
df = brut.to_pandas()
for colonne in ["statut", "categorie", "priorite", "satisfaction"]:
  print(df[colonne].value_counts(), "\n")
for message in df[df["canal"] == "formulaire"]["message"].head(3):
  print(message, "\n---")
```

### Solution

In [ ]:
brut = load_dataset(
  "csv", data_files="tickets.csv", split="train", keep_default_na=False
)
print(brut)
df = brut.to_pandas()
for colonne in ["statut", "categorie", "priorite", "satisfaction"]:
  print(df[colonne].value_counts(), "\n")
for message in df[df["canal"] == "formulaire"]["message"].head(3):
  print(message, "\n---")

Ce qu'on y trouve : des tickets encore ouverts ou fusionnés ; des étiquettes vides ou écrites de plusieurs façons (`Livraison `, `REPARATION`, `info`) ; du HTML dans les messages du formulaire ; des adresses e-mail et des numéros de téléphone ; des signatures de téléphone et des citations d'anciens e-mails ; des réponses vides ou inutilisables (`voir mail`, `test`) ; des clients mécontents de certaines réponses (satisfaction de 1 ou 2) ; des messages en anglais ; et, invisibles ici, des doublons et des réponses qui donnent l'ancien délai de retour, de 14 jours au lieu de 30.

## 2. Nettoyer les messages

Écrivez `nettoie(message)`, qui rend le texte d'un message tel que l'équipe l'a mis dans `test.jsonl` :

- le HTML retiré : `</p><p>` devient un saut de paragraphe, `<br>` un saut de ligne, les autres balises disparaissent, les entités (`&nbsp;`, `&#39;`) redeviennent des caractères, et l'espace insécable une espace ;
- la citation d'un ancien e-mail coupée, à partir de sa ligne `Le … a écrit :` ;
- la signature de téléphone retirée (une dernière ligne `Envoyé de …` ou `Envoyé depuis …`, parfois précédée de `--`) ;
- les adresses e-mail, numéros de téléphone et adresses postales remplacés par `[EMAIL]`, `[TELEPHONE]` et `[ADRESSE]` : c'est la pseudonymisation qu'impose le RGPD avant l'entraînement.

Écrivez aussi `normalise_etiquette(valeur)`, qui ramène une étiquette à sa forme de la charte : sans espaces autour, en minuscules, sans accents, `info` devenu `information` et `normal` devenu `normale`. Appliquez les deux à `brut` avec `map`. La cellule suivante vérifie le nettoyage : chaque ticket de `test.jsonl` doit retrouver, dans `propre`, exactement son message.

**Documentation utile**

- [`html.unescape`](https://docs.python.org/3/library/html.html#html.unescape)
- [`re.sub`](https://docs.python.org/3/library/re.html#re.sub)
- [`unicodedata.normalize`](https://docs.python.org/3/library/unicodedata.html#unicodedata.normalize)
- [`Dataset.map`](https://huggingface.co/docs/datasets/package_reference/main_classes#datasets.Dataset.map)

In [ ]:
# Votre code ici

### Conseils supplémentaires

- Les expressions régulières à utiliser :
  - les adresses e-mail : `r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+"`
  - les numéros : `r"(?:\+33 ?|0)[1-9](?:[ .-]?\d{2}){4}"`
  - les adresses : `r"\d+ (?:rue|avenue|boulevard|impasse|allée|place|chemin) [^,\n]+, \d{5} [^\s.,]+"`
  - la citation : `r"\n[^\n]* a écrit :\n.*"`, avec `re.DOTALL` pour que `.*` aille jusqu'à la fin
  - la signature : `r"\n(?:--\n)?Envoyé (?:de|depuis) [^\n]*$"`, sur le texte sans espaces de fin (`strip()`)
- Retirer les accents : `unicodedata.normalize("NFKD", texte)` sépare chaque lettre accentuée en sa lettre et son accent ; `unicodedata.combining(c)` est vrai pour un accent seul, à écarter
- `brut.map(fonction)` : `fonction` reçoit un ticket (un dictionnaire) et renvoie les colonnes à remplacer
- Exemple à vérifier : `nettoie("<p>Bonjour,</p><p>Mon numéro&nbsp;: 06 39 98 12 34</p>")` donne `"Bonjour,\n\nMon numéro : [TELEPHONE]"`

```python
EMAIL = re.compile(r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+")
TELEPHONE = re.compile(r"(?:\+33 ?|0)[1-9](?:[ .-]?\d{2}){4}")
ADRESSE = re.compile(
  r"\d+ (?:rue|avenue|boulevard|impasse|allée|place|chemin) [^,\n]+, \d{5} [^\s.,]+"
)
CITATION = re.compile(r"\n[^\n]* a écrit :\n.*", re.DOTALL)
SIGNATURE_MOBILE = re.compile(r"\n(?:--\n)?Envoyé (?:de|depuis) [^\n]*$")


def nettoie(message: str) -> str:
  texte = re.sub(r"</p>\s*<p>", "\n\n", message)
  texte = re.sub(r"<br\s*/?>", ..., texte)
  texte = re.sub(r"<[^>]+>", ..., texte)
  texte = html.unescape(texte).replace("\xa0", " ")
  texte = CITATION.sub("", texte)
  texte = SIGNATURE_MOBILE.sub("", texte.strip())
  texte = EMAIL.sub("[EMAIL]", texte)
  texte = TELEPHONE.sub(..., texte)
  texte = ADRESSE.sub(..., texte)
  return texte.strip()


def sans_accents(texte: str) -> str:
  texte = unicodedata.normalize("NFKD", texte)
  return "".join(c for c in texte if not ...)


def normalise_etiquette(valeur: str) -> str:
  valeur = sans_accents(valeur.strip().lower())
  return {"info": "information", "normal": "normale"}.get(valeur, valeur)


propre = brut.map(
  lambda t: {
    "message": nettoie(t["message"]),
    "categorie": ...,
    "priorite": ...,
  }
)
```

### Solution

In [ ]:
EMAIL = re.compile(r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+")
TELEPHONE = re.compile(r"(?:\+33 ?|0)[1-9](?:[ .-]?\d{2}){4}")
ADRESSE = re.compile(
  r"\d+ (?:rue|avenue|boulevard|impasse|allée|place|chemin) [^,\n]+, \d{5} [^\s.,]+"
)
CITATION = re.compile(r"\n[^\n]* a écrit :\n.*", re.DOTALL)
SIGNATURE_MOBILE = re.compile(r"\n(?:--\n)?Envoyé (?:de|depuis) [^\n]*$")


def nettoie(message: str) -> str:
  texte = re.sub(r"</p>\s*<p>", "\n\n", message)
  texte = re.sub(r"<br\s*/?>", "\n", texte)
  texte = re.sub(r"<[^>]+>", "", texte)
  texte = html.unescape(texte).replace("\xa0", " ")
  texte = CITATION.sub("", texte)
  texte = SIGNATURE_MOBILE.sub("", texte.strip())
  texte = EMAIL.sub("[EMAIL]", texte)
  texte = TELEPHONE.sub("[TELEPHONE]", texte)
  texte = ADRESSE.sub("[ADRESSE]", texte)
  return texte.strip()


def sans_accents(texte: str) -> str:
  texte = unicodedata.normalize("NFKD", texte)
  return "".join(c for c in texte if not unicodedata.combining(c))


def normalise_etiquette(valeur: str) -> str:
  valeur = sans_accents(valeur.strip().lower())
  return {"info": "information", "normal": "normale"}.get(valeur, valeur)


propre = brut.map(
  lambda t: {
    "message": nettoie(t["message"]),
    "categorie": normalise_etiquette(t["categorie"]),
    "priorite": normalise_etiquette(t["priorite"]),
  }
)

In [ ]:
messages_propres = dict(zip(propre["ticket_id"], propre["message"]))
identiques = sum(messages_propres[t["ticket_id"]] == t["message"] for t in test)
print(f"{identiques} messages de test sur {len(test)} retrouvés à l'identique")
print(propre.to_pandas()["categorie"].value_counts())

## 3. Écarter les tickets inutilisables

Écrivez `raison_rejet(ticket)`, qui renvoie pourquoi un ticket ne peut pas servir d'exemple, ou `None` s'il est bon. Dans cet ordre :

1. `"statut"` : le ticket n'est pas résolu (ouvert, ou fusionné avec un autre) ;
2. `"charte"` : sa réponse ne respecte pas la charte (`respecte_charte`, du TP précédent) : réponses vides ou inutilisables, l'agent qui tutoie, les réponses en anglais ;
3. `"satisfaction"` : le client a noté la réponse 1 ou 2 ;
4. `"etiquette"` : sa catégorie ou sa priorité, même normalisée, n'est pas une valeur de la charte (vide, le plus souvent) ;
5. `"ancienne politique"` : un ticket d'avant le 1er juin 2025 dont la réponse donne l'ancien délai de retour (`ANCIEN_DELAI`, ci-dessous) : il apprendrait au modèle une règle qui n'a plus cours. Attention, « 14 jours » seul ne suffit pas : le remboursement se fait toujours dans les 14 jours suivant le contrôle du retour.

Comptez les tickets par raison, puis gardez les bons avec `filter`.

**Documentation utile**

- [`Dataset.filter`](https://huggingface.co/docs/datasets/package_reference/main_classes#datasets.Dataset.filter)

In [ ]:
ANCIEN_DELAI = re.compile(r"14 jours (?:après|suivant) la livraison|pendant 14 jours")

In [ ]:
# Votre code ici

### Conseils supplémentaires

- Les dates sont des chaînes ISO (`"2025-05-30T14:12"`) : elles se comparent comme du texte, `ticket["date"] < "2025-06-01"`
- `satisfaction` est une chaîne : `"1"`, `"2"`… ou `""` quand le client n'a pas noté (pas une raison de l'écarter)
- `pd.Series([raison_rejet(t) for t in propre]).fillna("gardé").value_counts()`

```python
def raison_rejet(ticket: dict) -> str | None:
  if ticket["statut"] != "resolu":
    return "statut"
  if not respecte_charte(ticket["reponse"]):
    return ...
  if ticket["satisfaction"] in ...:
    return "satisfaction"
  if ticket["categorie"] not in CATEGORIES or ...:
    return "etiquette"
  if ticket["date"] < "2025-06-01" and ANCIEN_DELAI.search(...):
    return "ancienne politique"
  return None


print(pd.Series([raison_rejet(t) for t in propre]).fillna("gardé").value_counts())
gardes = propre.filter(lambda t: raison_rejet(t) is None)
print(len(gardes), "tickets gardés")
```

### Solution

In [ ]:
def raison_rejet(ticket: dict) -> str | None:
  if ticket["statut"] != "resolu":
    return "statut"
  if not respecte_charte(ticket["reponse"]):
    return "charte"
  if ticket["satisfaction"] in ("1", "2"):
    return "satisfaction"
  if ticket["categorie"] not in CATEGORIES or ticket["priorite"] not in PRIORITES:
    return "etiquette"
  if ticket["date"] < "2025-06-01" and ANCIEN_DELAI.search(ticket["reponse"]):
    return "ancienne politique"
  return None


print(pd.Series([raison_rejet(t) for t in propre]).fillna("gardé").value_counts())
gardes = propre.filter(lambda t: raison_rejet(t) is None)
print(len(gardes), "tickets gardés")

## 4. Dédupliquer

L'export a tourné deux fois certains jours (des doublons exacts), et des clients ont renvoyé leur message, précédé de `Relance :` ou suivi d'une ligne de plus. Écrivez `cle(message)`, une forme normalisée du message : en minuscules, sans accents, ses seuls mots et nombres séparés par une espace. Puis, en parcourant les tickets du plus ancien au plus récent (`sort("date")`), gardez un ticket seulement si sa clé **ne contient** la clé d'aucun ticket déjà gardé : une relance contient le message d'origine. Combien de tickets partent ?

Sur un gros corpus, on compare plutôt des empreintes (MinHash), car tester chaque paire coûte trop cher ; ici, 1 600 messages se comparent en quelques secondes.

**Documentation utile**

- [`re.findall`](https://docs.python.org/3/library/re.html#re.findall)
- [`Dataset.sort`](https://huggingface.co/docs/datasets/package_reference/main_classes#datasets.Dataset.sort)

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `re.findall(r"[a-z0-9]+", texte)` : les mots et nombres d'un texte en minuscules, sans accents (`sans_accents`, de l'exercice 2)
- Exemple à vérifier : `cle("Relance : Où est ma commande CMD-123456 ?")` donne `"relance ou est ma commande cmd 123456"`
- Une liste `cles_gardees`, remplie par la fonction passée à `filter` : `any(ancienne in nouvelle for ancienne in cles_gardees)`

```python
def cle(message: str) -> str:
  return " ".join(re.findall(r"[a-z0-9]+", sans_accents(message.lower())))


cles_gardees = []


def est_nouveau(ticket: dict) -> bool:
  nouvelle = cle(ticket["message"])
  if any(... in nouvelle for ancienne in cles_gardees):
    return False
  cles_gardees.append(nouvelle)
  return True


uniques = gardes.sort("date").filter(est_nouveau)
print(len(gardes) - len(uniques), "doublons retirés,", len(uniques), "tickets restent")
```

### Solution

In [ ]:
def cle(message: str) -> str:
  return " ".join(re.findall(r"[a-z0-9]+", sans_accents(message.lower())))


cles_gardees = []


def est_nouveau(ticket: dict) -> bool:
  nouvelle = cle(ticket["message"])
  if any(ancienne in nouvelle for ancienne in cles_gardees):
    return False
  cles_gardees.append(nouvelle)
  return True


uniques = gardes.sort("date").filter(est_nouveau)
print(len(gardes) - len(uniques), "doublons retirés,", len(uniques), "tickets restent")

## 5. Retirer les fuites vers le test

Les 150 tickets de `test.jsonl` viennent du même export : s'ils restent dans le jeu d'entraînement, le modèle les aura vus, et la mesure finale sera trop belle. Retirez d'`uniques` les tickets dont l'identifiant est dans le test, puis ceux dont la clé contient la clé d'un message de test, ou y est contenue (une relance d'un ticket de test, ou le même message d'un autre client). Comptez ce que chaque étape retire : l'identifiant suffit-il ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

```python
ids_test = set(test["ticket_id"])
cles_test = [cle(m) for m in test["message"]]


def ressemble_au_test(ticket: dict) -> bool:
  k = cle(ticket["message"])
  return any(c in k or ... for c in cles_test)


sans_ids = uniques.filter(lambda t: t["ticket_id"] not in ...)
entrainement = sans_ids.filter(lambda t: not ressemble_au_test(t))
print(len(uniques) - len(sans_ids), "retirés par identifiant")
print(len(sans_ids) - len(entrainement), "retirés par ressemblance")
```

### Solution

In [ ]:
ids_test = set(test["ticket_id"])
cles_test = [cle(m) for m in test["message"]]


def ressemble_au_test(ticket: dict) -> bool:
  k = cle(ticket["message"])
  return any(c in k or k in c for c in cles_test)


sans_ids = uniques.filter(lambda t: t["ticket_id"] not in ids_test)
entrainement = sans_ids.filter(lambda t: not ressemble_au_test(t))
print(len(uniques) - len(sans_ids), "retirés par identifiant")
print(len(sans_ids) - len(entrainement), "retirés par ressemblance")

## 6. Mettre au format de `trl` et découper

Chaque ticket devient un exemple au format conversationnel prompt-complétion de `trl` : un `prompt` de deux messages (`CONSIGNES` en `system`, le message du client en `user`) et une `completion` d'un message `assistant`, l'objet JSON attendu. Écrivez `vers_exemple(ticket)`, appliquez-la à `entrainement` et à `test` en ne gardant que ces deux colonnes, plus `categorie`. Puis coupez l'entraînement en entraînement et validation (10 %), stratifiés par catégorie, et réunissez les trois parties dans un `DatasetDict`. Regardez la répartition par catégorie : faut-il rééquilibrer ?

**Documentation utile**

- [Les formats de `trl`](https://huggingface.co/docs/trl/dataset_formats#prompt-completion)
- [`Dataset.class_encode_column`](https://huggingface.co/docs/datasets/package_reference/main_classes#datasets.Dataset.class_encode_column)
- [`Dataset.train_test_split`](https://huggingface.co/docs/datasets/package_reference/main_classes#datasets.Dataset.train_test_split)
- [`ClassLabel`](https://huggingface.co/docs/datasets/package_reference/main_classes#datasets.ClassLabel)
- [`DatasetDict`](https://huggingface.co/docs/datasets/package_reference/main_classes#datasets.DatasetDict)

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `json.dumps({...}, ensure_ascii=False)` : le JSON attendu, accents compris
- `map(vers_exemple, remove_columns=...)` : `remove_columns` retire les colonnes d'origine, sauf celles que `vers_exemple` renvoie (`categorie`)
- `train_test_split(test_size=0.1, seed=0, stratify_by_column="categorie")` demande une colonne de classes : `class_encode_column("categorie")` d'abord
- Après `class_encode_column`, les catégories sont des numéros : `exemples.features["categorie"]` (un `ClassLabel`) les traduit (`int2str`), et `cast_column` donne la même colonne au test
- `pd.Series(...).value_counts()` : la répartition

```python
def vers_exemple(ticket: dict) -> dict:
  attendu = {
    "categorie": ticket["categorie"],
    "priorite": ticket["priorite"],
    "reponse": ticket["reponse"],
  }
  return {
    "prompt": [
      {"role": "system", "content": CONSIGNES},
      {"role": "user", "content": ...},
    ],
    "completion": [{"role": "assistant", "content": json.dumps(...)}],
    "categorie": ticket["categorie"],
  }


exemples = entrainement.map(vers_exemple, remove_columns=entrainement.column_names)
exemples_test = test.map(vers_exemple, remove_columns=test.column_names)
exemples = exemples.class_encode_column("categorie")
classes = exemples.features["categorie"]
exemples_test = exemples_test.cast_column("categorie", classes)
decoupe = exemples.train_test_split(...)
jeux = DatasetDict(
  {"train": decoupe["train"], "validation": decoupe["test"], "test": exemples_test}
)
print(jeux)
print(pd.Series(classes.int2str(...)).value_counts())
```

### Solution

In [ ]:
def vers_exemple(ticket: dict) -> dict:
  attendu = {
    "categorie": ticket["categorie"],
    "priorite": ticket["priorite"],
    "reponse": ticket["reponse"],
  }
  return {
    "prompt": [
      {"role": "system", "content": CONSIGNES},
      {"role": "user", "content": ticket["message"]},
    ],
    "completion": [
      {"role": "assistant", "content": json.dumps(attendu, ensure_ascii=False)}
    ],
    "categorie": ticket["categorie"],
  }


exemples = entrainement.map(vers_exemple, remove_columns=entrainement.column_names)
exemples_test = test.map(vers_exemple, remove_columns=test.column_names)
exemples = exemples.class_encode_column("categorie")
classes = exemples.features["categorie"]
exemples_test = exemples_test.cast_column("categorie", classes)
decoupe = exemples.train_test_split(
  test_size=0.1, seed=0, stratify_by_column="categorie"
)
jeux = DatasetDict(
  {"train": decoupe["train"], "validation": decoupe["test"], "test": exemples_test}
)
print(jeux)
print(pd.Series(classes.int2str(jeux["train"]["categorie"])).value_counts())

`compte` est huit fois plus rare qu'`information` : avec une quarantaine d'exemples, il reste appris (les demandes y sont peu variées). On ne rééquilibre que si la mesure, par catégorie, montre une catégorie rare mal traitée, et alors plutôt en ajoutant des exemples qu'en dupliquant ceux qu'on a.

## 7. Enregistrer et publier

Enregistrez `jeux` sur le disque (`save_to_disk`), puis chaque partie en JSON Lines (`to_json`) : c'est ce que chargera le TP d'entraînement. Puis, si vous avez un compte Hugging Face, publiez le jeu sur le Hub, en **privé**, et étiquetez cette version `v1.0` : le modèle entraîné dessus pourra dire exactement quelles données il a vues. Il faut un jeton d'accès en écriture, rangé dans les secrets de Colab sous le nom `HF_TOKEN` (la clé dans la barre de gauche) ; sans lui, la cellule ne publie rien.

**Documentation utile**

- [`DatasetDict.save_to_disk`](https://huggingface.co/docs/datasets/package_reference/main_classes#datasets.DatasetDict.save_to_disk)
- [`Dataset.to_json`](https://huggingface.co/docs/datasets/package_reference/main_classes#datasets.Dataset.to_json)
- [`DatasetDict.push_to_hub`](https://huggingface.co/docs/datasets/package_reference/main_classes#datasets.DatasetDict.push_to_hub)
- [`HfApi.create_tag`](https://huggingface.co/docs/huggingface_hub/package_reference/hf_api#huggingface_hub.HfApi.create_tag)

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `jeux.save_to_disk("support-sft")`, et `jeux[partie].to_json(f"support-sft-{partie}.jsonl", force_ascii=False)` pour chaque partie
- Le jeton : `os.environ.get("HF_TOKEN")`, ou, dans Colab, `google.colab.userdata.get("HF_TOKEN")`, qui lève une exception quand le secret n'existe pas
- `HfApi(token=jeton).whoami()["name"]` : votre nom d'utilisateur, pour nommer le dépôt `<nom>/support-sft`

```python
import os

jeux.save_to_disk("support-sft")
for partie in jeux:
  jeux[partie].to_json(f"support-sft-{partie}.jsonl", force_ascii=False)

jeton = os.environ.get("HF_TOKEN")
try:
  from google.colab import userdata

  jeton = userdata.get("HF_TOKEN")
except Exception:
  pass
if jeton:
  from huggingface_hub import HfApi

  api = HfApi(token=jeton)
  depot = f"{api.whoami()['name']}/support-sft"
  jeux.push_to_hub(depot, private=..., token=jeton)
  api.create_tag(depot, tag=..., repo_type="dataset", exist_ok=True)
  print("Publié :", depot)
else:
  print("Pas de jeton HF_TOKEN : rien n'est publié")
```

### Solution

In [ ]:
import os

jeux.save_to_disk("support-sft")
for partie in jeux:
  jeux[partie].to_json(f"support-sft-{partie}.jsonl", force_ascii=False)

jeton = os.environ.get("HF_TOKEN")
try:
  from google.colab import userdata

  jeton = userdata.get("HF_TOKEN")
except Exception:
  pass
if jeton:
  from huggingface_hub import HfApi

  api = HfApi(token=jeton)
  depot = f"{api.whoami()['name']}/support-sft"
  jeux.push_to_hub(depot, private=True, token=jeton)
  api.create_tag(depot, tag="v1.0", repo_type="dataset", exist_ok=True)
  print("Publié :", depot)
else:
  print("Pas de jeton HF_TOKEN : rien n'est publié")

## À retenir

- Sur environ 2 400 tickets exportés, un peu plus de la moitié devient un exemple : la curation coûte plus d'exemples qu'on ne l'imagine, et c'est elle qui décide de ce que le modèle apprendra (le ton de l'agent qui tutoie, l'ancien délai de retour).
- Les fuites passent par les doublons : retirer les identifiants du test ne suffit pas.
- Le jeu est versionné : `v1.0` sur le Hub, ou ces fichiers JSON Lines, à citer avec le modèle qu'on en tirera.